# 06 · Model analysis

Which factors matter, would more data help, where does the model work best and where does it fail? Repeated 5-fold grouped cross-validation. Code: `evaluation/analysis.py`. Takes about 30 seconds.

> **Privacy:** this notebook shows only counts, metrics, charts and study codes (R0001, Q0001), never NEBSL records. **Before committing, clear all outputs** (VS Code: `...` → *Clear All Outputs* → save).

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make `recommendation` importable whether this runs from notebooks/, recommendation/ or ml/
ML = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "recommendation" / "config.py").exists())
sys.path.insert(0, str(ML))

import pandas as pd
from IPython.display import Image, display
from recommendation.config import load_config

pd.set_option("display.width", 160)
cfg = load_config()
DATASET = cfg["paths"]["matching_dataset"]
OUTPUTS = Path(cfg["paths"]["outputs"])
print("Dataset:", Path(DATASET).name)

In [ ]:
import joblib
from recommendation.evaluation import analysis as A
from recommendation.matching.mcda import Weights, score

model_file = Path(cfg["paths"]["models"]) / "matching_ranker.joblib"
settings = joblib.load(model_file)["settings"] if model_file.exists() else {"model": "logistic", "C": 1.0}
pairs = pd.read_excel(DATASET, sheet_name="Ranking_Pairs")
scored = score(pairs, Weights(**cfg["matching"]["weights"]))
kw = {"repeats": 5, "seed": cfg["split"]["seed"]}

## 1. Ablation: remove one factor at a time

In [ ]:
A.ablation(scored, settings, **kw).round(3)

## 2. Learning curve: does more data help?

In [ ]:
curve = A.learning_curve(scored, settings, **kw)
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(curve["train_requests"], curve["ml_mrr"], marker="o", label="ML ranker")
ax.plot(curve["train_requests"], curve["mcda_mrr"], linestyle="--", label="MCDA (no training)")
ax.set_xlabel("training requests"); ax.set_ylabel("MRR"); ax.legend(); ax.set_title("Learning curve", loc="left")
plt.show()
curve.round(3)

## 3. Subgroups: ML vs MCDA

In [ ]:
ranks = A.cv_ranks(scored, settings, **kw)
A.subgroups(scored, ranks).round(3)

## 4. Errors: requests where the chosen cornea ranked below #5

In [ ]:
err = A.errors(scored, ranks, settings)
print(f"{err['missed']} of {err['requests']} requests ({err['missed_share']:.0%})")
err["comparison"].round(2)

Where the model misses, the chosen cornea was often a lower grade and older than the model's #1, which suggests information not in the data (surgeon preference, timing, logistics).